In [32]:
from pathlib import Path

from torchvision.models import resnet18, ResNet18_Weights
import json

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F

import cv2

from torch.utils.data import Dataset, DataLoader

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Device:", device)

Device: cpu


In [108]:
# Training config

config = {
    "run_name": "all_videos_resnet18_v1",
    "epochs": 3,
    "chunk_size": 16,
    "max_image_size": 128,
    "learning_rate": 0.001,
    "hidden_size": 128,
    "initialization": "pretrained",
    "checkpoint": None,
}


In [106]:
DATA_FOLDER = Path("data") / "singleObjectDetection"

PROJECT_ROOT = next(
    path
    for path in [Path.cwd(), *Path.cwd().parents]
    if (path / DATA_FOLDER).is_dir()
)

DATA_ROOT = PROJECT_ROOT / DATA_FOLDER


In [107]:
class RotationPositionDataset(Dataset):

    def __init__(
        self,
        split_dir,
        position_file="positions.csv",
        max_side=128,
        transform=None
    ):
        self.csv_files = sorted(
            Path(split_dir).glob("*/rotation_labels/rotation_*.csv")
        )

        self.position_file = position_file
        self.max_side = max_side
        self.transform = transform

    def __len__(self):
        return len(self.csv_files)

    def __getitem__(self, idx):

        csv_path = self.csv_files[idx]
        video_dir = csv_path.parent.parent

        rotations = pd.read_csv(csv_path)

        # Match object positions to rotation labels by filename
        positions = (
            pd.read_csv(video_dir / self.position_file)
            .set_index("filename")
            .reindex(rotations["filename"])
        )

        xy = positions[["cx", "cy"]].to_numpy(dtype=np.float32)

        images = []
        original_xy = []
        rotated_xy = []
        valid = []

        for i, row in enumerate(rotations.itertuples(index=False)):

            image = cv2.imread(str(video_dir / "input" / row.filename))
            h, w = image.shape[:2]

            M = cv2.getRotationMatrix2D(
                (w / 2, h / 2),
                row.angle_deg,
                row.zoom
            )

            # Rotate image
            image = cv2.warpAffine(image, M, (w, h))

            # Rotate object position using the same matrix
            point = xy[i]
            point_rot = M @ np.array([point[0], point[1], 1.0])

            # Normalize positions to image coordinates [0,1]
            original_xy.append(point / [w, h])
            rotated_xy.append(point_rot / [w, h])

            valid.append(
                np.isfinite(point_rot).all()
                and 0 <= point_rot[0] < w
                and 0 <= point_rot[1] < h
            )

            # Resize image
            scale = self.max_side / max(h, w)
            new_size = (
                max(8, round(w * scale / 8) * 8),
                max(8, round(h * scale / 8) * 8)
            )

            image = cv2.resize(image, new_size)
            images.append(cv2.cvtColor(image, cv2.COLOR_BGR2RGB))

        frames = torch.from_numpy(
            np.stack(images)
        ).permute(0, 3, 1, 2).float() / 255.0

        return {
            "frames": frames,
            "angles": torch.tensor(
                rotations["angle_deg"].to_numpy(), dtype=torch.float32
            ),
            "pos_original": torch.tensor(original_xy, dtype=torch.float32),
            "pos_rotated": torch.tensor(rotated_xy, dtype=torch.float32),
            "pos_valid": torch.tensor(valid, dtype=torch.bool),
            "zoom": torch.tensor(
                rotations["zoom"].to_numpy(), dtype=torch.float32
            )
        }


In [90]:
train_loader = DataLoader(
    RotationPositionDataset(
        DATA_ROOT / "train",
        "positions.csv",
    ),
    batch_size=1,
    shuffle=True,
)

val_loader = DataLoader(
    RotationPositionDataset(
        DATA_ROOT / "val",
        "positions.csv",
    ),
    batch_size=1,
    shuffle=False,
)

test_loader = DataLoader(
    RotationPositionDataset(
        DATA_ROOT / "test",
        "positions.csv",
    ),
    batch_size=1,
    shuffle=False,
)

In [36]:
def rotation_loss(predictions, true_angles):
    radians = torch.deg2rad(true_angles)

    targets = torch.stack([
        torch.cos(radians),
        torch.sin(radians),
        ],
      dim=1,
     )

    similarity = (
        predictions*targets
    ).sum(dim=1)
    
    return (1-similarity).mean() 
    # error of 90 deg: loss 1
    # error of 180 deg: loss 2
    
    

In [83]:
class RotationCNNGRU(nn.Module):

    def __init__(self, hidden_size=128):
        super().__init__()

        backbone = resnet18(
            weights = ResNet18_Weights.DEFAULT
        )

        self.encoder = nn.Sequential(
            *list(backbone.children())[:-2] # remove last 2 layers: average pooling and classification
        )
        for param in self.encoder.parameters():
            param.requires_grad=False # freeze all remaining layers

        self.spatial_pool = nn.AdaptiveAvgPool2d((2, 3)) # changes every feature to [512, 2, 3]

        feature_dim = 512 * 2 * 3

        self.projection = nn.Sequential(
            nn.Linear(feature_dim * 2, 128),
            nn.ReLU(),
            nn.Dropout(0.1)
        ) # converts large comparison into 128 features per frame. Before entering GRU, it has shape [T, 128]

        # GRU: model rotation over time
        self.gru = nn.GRU(
            input_size=128,
            hidden_size=hidden_size,
            batch_first=True
        )
        # gru output: [1, T, hidden_size]
        

        # Output: cosine and sine of rotation angle
        self.head = nn.Linear(hidden_size, 2) # angle head

        self.register_buffer(
            "mean",
            torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1)
        ) # image normalization

        self.register_buffer(
            "std",
            torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1)
        ) # image normalization

    def forward(self, frames, reference, hidden=None):
        # frames: [T, 3, H, W]
        # reference: [3, H, W]

        T = frames.shape[0]

        self.encoder.eval()

        frames = (frames - self.mean) / self.std # normalization

        reference = reference.unsqueeze(0)
        reference = (reference - self.mean) / self.std

        reference_features = self.encoder(reference)
        frame_features = self.encoder(frames)

        reference_features = self.spatial_pool(
            reference_features
        ).flatten(1)

        frame_features = self.spatial_pool(
            frame_features
        ).flatten(1)

        # Compare current frame with reference frame
        reference_features = reference_features.expand(T, -1)

        difference = frame_features - reference_features

        combined = torch.cat(
            [frame_features, difference],
            dim=1
        )

        features = self.projection(combined)

        # Process chronological sequence
        features = features.unsqueeze(0)

        output, hidden = self.gru(
            features,
            hidden
        )

        predictions = self.head(
            output.squeeze(0)
        )

        predictions = F.normalize(
            predictions,
            dim=-1,
            eps=1e-6
        )

        return predictions, hidden

In [84]:
def run_trajectory( # keep whole trajectory on CPU and move chunks to GPU
    frames, angles, model, device, chunk_size=16, optimizer=None
):
    # trains model no recived chunks 
    training=optimizer is not None

    model.train(training)

    reference = frames[0].to(device)

    hidden = None
    total_loss = 0.0 
    number_of_frames = len(frames)

    with torch.set_grad_enabled(training):
        for start in range(0, number_of_frames, chunk_size):
            end = min(start+chunk_size, number_of_frames)
            frame_chunk = frames[start:end].to(device)
            angle_chunk = angles[start:end].to(device)
            if training:
                optimizer.zero_grad(set_to_none=True)
            
            predictions, hidden = model( # get predictions
                frames=frame_chunk,
                reference=reference,
                hidden=hidden
            )

            loss = rotation_loss( # get loss
                predictions = predictions,
                true_angles = angle_chunk
            )

            if training:
                loss.backward()

                nn.utils.clip_grad_norm_(
                    (
                    parameter
                    for parameter in model.parameters()
                    if parameter.requires_grad
                    ),
                    max_norm=1.0
                )
                optimizer.step()

            hidden = hidden.detach()

            chunk_length = end-start
            total_loss = total_loss + (loss.item()*chunk_length)
    return total_loss/number_of_frames # loss per frame
        

In [86]:
model = RotationCNNGRU(
    hidden_size=config["hidden_size"]
).to(device)

optimizer = torch.optim.Adam(
    (
        parameter
        for parameter in model.parameters()
        if parameter.requires_grad
    ),
    lr=config["learning_rate"],
)

In [100]:
def run_loader(loader, model, device, chunk_size, optimizer=None, max_trajectories=None):
    # Wraps run_trajectory and dataloader. tun this function to train model
    total_loss = 0.0
    total_frames = 0

    for trajectory_index, batch in enumerate(loader):
        if (max_trajectories is not None and trajectory_index >= max_trajectories):
            break
        frames = batch["frames"].squeeze(0)
        angles = batch["angles"].squeeze(0)

        trajectory_loss = run_trajectory(
            frames=frames,
            angles=angles,
            model=model,
            device=device,
            chunk_size=chunk_size,
            optimizer=optimizer,
        )

        number_of_frames=len(frames)
        total_loss = total_loss + trajectory_loss*number_of_frames
        total_frames = total_frames + number_of_frames
    return total_loss/total_frames

In [ ]:
history = []


for epoch in range(config["epochs"]):

    train_loss = run_loader(
        loader=train_loader,
        model=model,
        device=device,
        chunk_size=config["chunk_size"],
        optimizer=optimizer,
    )

    val_loss = run_loader(
        loader=val_loader,
        model=model,
        device=device,
        chunk_size=config["chunk_size"],
        optimizer=None,
    )

    history.append({
        "epoch": epoch + 1,
        "train_loss": train_loss,
        "val_loss": val_loss,
    })

    print(
        f"Epoch {epoch + 1}/{test_epochs} | "
        f"Train: {train_loss:.4f} | "
        f"Validation: {val_loss:.4f}"
    )

In [ ]:
history_df = pd.DataFrame(history)

plt.plot(
    history_df["epoch"],
    history_df["train_loss"],
    marker="o",
    label="Training",
)

plt.plot(
    history_df["epoch"],
    history_df["val_loss"],
    marker="o",
    label="Validation",
)

plt.xlabel("Epoch")
plt.ylabel("Circular rotation loss")
plt.legend()
plt.grid(True)
plt.show()